# Milestone 1 — Auditable Data and Retrieval

| Required evidence | Section |
|---|---|
| Validated training and evaluation records | 1 |
| Sample chunks with source metadata | 2–3 |
| Retrieval results for supported and unsupported questions | 4–5 |
| A documented threshold hypothesis | 6–7 |

All logic lives in `src/datasets.py` and `src/retrieval.py`; this notebook runs and inspects it.

In [ ]:
import json
import os
from pathlib import Path

# Run from the project root so data/ and src/ resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.datasets import load_evaluation, load_training, validate
from src.retrieval import Retriever, build_chunks, evaluate_retrieval, load_documents

print("Working directory:", Path.cwd())

## 1. Data validation

Checks run before training:

- every training example has `id`, `context`, `question`, `answer`, and `reference_ids`,
- every `reference_id` and every `[source]` cited in an answer exists in the knowledge base,
- training and evaluation IDs are unique,
- no evaluation question appears in the training set (exact match = error, high word overlap = warning).

Errors block training; warnings are reviewed by hand.

In [ ]:
training = load_training()
evaluation = load_evaluation()
report = validate(training=training, evaluation=evaluation)

print("Knowledge-base source IDs:", report.source_ids)
print("Training examples:        ", report.training_count)
print("Evaluation cases:         ", report.evaluation_count)
print("Valid:                    ", report.ok)
print()
print("Errors:")
for error in report.errors or ["(none)"]:
    print("  -", error)
print("Warnings:")
for warning in report.warnings or ["(none)"]:
    print("  -", warning)

In [ ]:
# Training examples per cited policy, and evaluation cases per expected source.
from collections import Counter

train_sources = Counter(s for e in training for s in e["reference_ids"])
eval_sources = Counter(s for c in evaluation for s in c["expected_sources"])
print(f"{'source':<18} {'train':>5} {'eval':>5}")
for source_id in report.source_ids:
    print(f"{source_id:<18} {train_sources[source_id]:>5} {eval_sources[source_id]:>5}")
print(f"{'(abstain cases)':<18} {'':>5} {sum(c['should_abstain'] for c in evaluation):>5}")

**Data-quality decisions**

- The supplied data was **not changed**.
- `train-style-02` has no `[source]` citation. This is intentional: it teaches the model to say the evidence is insufficient and escalate, so a citation would be wrong. It is kept as a warning, not an error.
- Evaluation cases stay in `data/evaluation/cases.json` and are never added to training.

## 2. Policy sources

In [ ]:
documents = load_documents()
for document_id, text in documents.items():
    print(f"{document_id:<18} {len(text):>5} chars")

## 3. Chunks with source metadata

Each chunk keeps `document_id`, `title`, `position`, and `text`, and the same fields are stored as the Qdrant payload so citations can be traced back to the source file.

In [ ]:
MAX_CHARS = 600  # try 250 for one chunk per paragraph

chunks = build_chunks(max_chars=MAX_CHARS)
print(f"{len(chunks)} chunks from {len({c.document_id for c in chunks})} documents")
print()
for chunk in chunks:
    print(f"[{chunk.chunk_id}] {chunk.title} (position {chunk.position}, {len(chunk.text)} chars)")
    print(f"    {chunk.text[:120]}...")
    print()

In [ ]:
retriever = Retriever()
retriever.index(chunks)

print("Embedding model:", retriever.embedding_model)
print("Dimension:      ", retriever.dimension)
print("top_k:          ", retriever.top_k)
print("min_score:      ", retriever.min_score)
print()

# Read one point back from Qdrant to show the metadata survived storage.
from src.retrieval import COLLECTION_NAME

point = retriever.client.scroll(COLLECTION_NAME, limit=1, with_payload=True)[0][0]
print("Stored payload:", json.dumps(point.payload, indent=2)[:400])

## 4. Example retrievals

One supported question and one the policies do not cover. The unsupported question should abstain instead of using the nearest weak match.

In [ ]:
def show(question):
    result = retriever.retrieve(question)
    print("Q:", question)
    print("Decision:", "ABSTAIN" if result.abstain else "ANSWER", "| sources:", result.source_ids)
    for hit in retriever.search(question):
        kept = "kept   " if hit.score >= retriever.min_score else "dropped"
        print(f"   {hit.score:.3f} {kept} [{hit.chunk.chunk_id}] {hit.chunk.text[:80]}...")
    print()

show("A digital workshop starts in 12 hours. Can the customer still receive a refund?")
show("What is on the cafeteria menu today?")

## 5. Retrieval on the held-out evaluation cases

In [ ]:
supported = [c for c in evaluation if not c["should_abstain"]]
unsupported = [c for c in evaluation if c["should_abstain"]]
print(f"{len(evaluation)} cases: {len(supported)} supported, {len(unsupported)} unsupported")
print()

retrieval_report = evaluate_retrieval(retriever, evaluation)
print(f"recall@{retrieval_report['top_k']}:         {retrieval_report['recall_at_k']:.3f}")
print(f"MRR:               {retrieval_report['mrr']:.3f}")
print(f"decision accuracy: {retrieval_report['decision_accuracy']:.3f}  (min_score={retrieval_report['min_score']})")
print()

print(f"{'case':<30} {'score':>6}  {'should':<8} {'got':<8} retrieved")
for row in retrieval_report["per_case"]:
    should = "abstain" if row["should_abstain"] else "answer"
    got = "abstain" if row["abstained"] else "answer"
    mark = "" if should == got else "  <-- wrong"
    print(f"{row['id']:<30} {row['top_score']:>6.3f}  {should:<8} {got:<8} {row['retrieved']}{mark}")

## 6. Threshold evidence

The best-match score per case shows whether a similarity threshold can separate supported from unsupported questions.

In [ ]:
top_scores = {row["id"]: row["top_score"] for row in retrieval_report["per_case"]}
sup_scores = sorted(top_scores[c["id"]] for c in supported)
unsup_scores = sorted(top_scores[c["id"]] for c in unsupported)

print("Supported top scores:  ", sup_scores)
print("Unsupported top scores:", unsup_scores)
print()
print(f"Lowest supported score:    {min(sup_scores):.3f}")
print(f"Highest unsupported score: {max(unsup_scores):.3f}")
if max(unsup_scores) < min(sup_scores):
    print("A threshold between these values separates the two groups perfectly.")
else:
    print("The groups overlap: no threshold separates them perfectly.")

In [ ]:
# Sweep thresholds and count correct answer/abstain decisions.
print(f"{'threshold':>9}  {'accuracy':>8}  {'missed answers':>14}  {'wrong answers':>13}")
for threshold in [0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    missed = sum(s < threshold for s in sup_scores)    # supported but abstained
    wrong = sum(s >= threshold for s in unsup_scores)  # unsupported but answered
    accuracy = (len(evaluation) - missed - wrong) / len(evaluation)
    print(f"{threshold:>9.2f}  {accuracy:>8.3f}  {missed:>14}  {wrong:>13}")

## 7. Threshold hypothesis

*Numbers below are from the run with `all-MiniLM-L6-v2`, `top_k=3`, 600-character chunks. Re-check them if you change any of these.*

**Hypothesis:** a cosine-similarity threshold of **`MIN_SCORE = 0.30`** rejects off-topic questions without refusing any supported question, but it cannot reject near-domain questions that the policies do not answer.

**Evidence**

- Off-topic questions (`eval-cafeteria`, `eval-bicycle`) score about **0.11**, far below every supported question.
- The weakest supported question (`eval-duplicate-payment`) scores **0.399**. A threshold of 0.40 or higher starts refusing supported questions.
- Thresholds from 0.20 to 0.35 all reach the best decision accuracy (**0.857**). 0.30 sits in the middle of that range, leaving margin on both sides.
- Retrieval ranking is perfect: recall@3 = 1.000 and MRR = 1.000.

**Known limitation**

- `eval-bank-chargeback` (0.592) and `eval-shipping-compensation` (0.566) are unsupported but score **higher** than several supported questions. They match the right topic (refunds, shipping), but the policy does not answer the specific request.
- No threshold can fix this: removing them needs a threshold above 0.592, which would refuse 6 of 10 supported questions.
- These two cases therefore must be handled at **generation time**: the model has to say the evidence is insufficient. This is a retrieval error that limits baseline RAG and hybrid RAG + LoRA equally, and it is a useful test of whether the adapter (trained on `train-style-02`-type examples) improves abstention.

**How to test the hypothesis further**

- More unsupported near-domain questions would show whether the 0.30 threshold holds beyond 4 cases.
- Paragraph-sized chunks (`MAX_CHARS = 250`) may lower scores for questions that only share a topic with a document.